In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)   # empty slices where a whole row was dropped
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

DATA = '../data'
X_train = np.load(f'{DATA}/train_cadences.npy', mmap_mode='r')   # (18000, 6, 16, 128) uint8
X_test  = np.load(f'{DATA}/test_cadences.npy', mmap_mode='r')    # (10000, 6, 16, 128) uint8
train = pd.read_csv(f'{DATA}/train_meta.csv')
test  = pd.read_csv(f'{DATA}/test_meta.csv')
print(X_train.shape, X_test.shape)
print(train.label.value_counts().to_dict())

(18000, 6, 16, 128) (10000, 6, 16, 128)
{'noise': 4211, 'rfi_persistent': 2892, 'rfi_intermittent': 2662, 'radar_chirp': 1849, 'satellite_pass': 1805, 'pulsar': 1629, 'scintillating_star': 1431, 'frb': 1042, 'eti_candidate': 479}


In [2]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode the labels in the training dataframe.
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(train["label"])

# Recreate E5's exact split.
indices = np.arange(len(y_encoded))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

print("Classes:", list(label_encoder.classes_))
print("Training samples:", len(train_idx))
print("Validation samples:", len(val_idx))

Classes: ['eti_candidate', 'frb', 'noise', 'pulsar', 'radar_chirp', 'rfi_intermittent', 'rfi_persistent', 'satellite_pass', 'scintillating_star']
Training samples: 14400
Validation samples: 3600


In [3]:
import numpy as np

def prepare_cnn_input_e8(X):
    X = np.asarray(X, dtype=np.float32)

    # Shape: (N, 6, 16, 128)
    mask = X != 0

    # Count valid frequency channels in each cadence.
    count = mask.sum(axis=3, keepdims=True)
    safe_count = np.maximum(count, 1)

    # Compute each cadence's mean, ignoring missing pixels.
    observed = np.where(mask, X, 0.0)
    mean = observed.sum(axis=3, keepdims=True) / safe_count

    centered = np.where(mask, X - mean, 0.0)

    # Compute each cadence's standard deviation.
    variance = (
        (centered ** 2).sum(axis=3, keepdims=True)
        / safe_count
    )
    std = np.sqrt(variance)

    # Avoid division by zero or amplifying nearly constant rows.
    std = np.maximum(std, 1.0)

    normalized = np.where(mask, centered / std, 0.0)

    # Preserve missing-pixel information, as in E5.
    result = np.concatenate(
        [normalized, mask.astype(np.float32)],
        axis=1
    )

    return result.astype(np.float32)

In [4]:
X_sample_e8 = prepare_cnn_input_e8(X_train[:32])

print("Input shape:", X_sample_e8.shape)
print("Data type:", X_sample_e8.dtype)
print("All finite:", np.isfinite(X_sample_e8).all())
print("Signal channels:", X_sample_e8[:, :6].shape)
print("Mask channels:", X_sample_e8[:, 6:].shape)

assert X_sample_e8.shape == (32, 12, 16, 128)
assert np.isfinite(X_sample_e8).all()
assert np.isin(X_sample_e8[:, 6:], [0.0, 1.0]).all()

print("E8 preprocessing checks passed.")

Input shape: (32, 12, 16, 128)
Data type: float32
All finite: True
Signal channels: (32, 6, 16, 128)
Mask channels: (32, 6, 16, 128)
E8 preprocessing checks passed.


In [5]:
X_train_split_e8 = prepare_cnn_input_e8(X_train[train_idx])
X_val_e8 = prepare_cnn_input_e8(X_train[val_idx])

print("Training:", X_train_split_e8.shape)
print("Validation:", X_val_e8.shape)

Training: (14400, 12, 16, 128)
Validation: (3600, 12, 16, 128)


In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

X_train_tensor_e8 = torch.from_numpy(X_train_split_e8)
X_val_tensor_e8 = torch.from_numpy(X_val_e8)

y_train_tensor_e8 = torch.tensor(
    y_encoded[train_idx], dtype=torch.long
)
y_val_tensor_e8 = torch.tensor(
    y_encoded[val_idx], dtype=torch.long
)

train_loader_e8 = DataLoader(
    TensorDataset(X_train_tensor_e8, y_train_tensor_e8),
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader_e8 = DataLoader(
    TensorDataset(X_val_tensor_e8, y_val_tensor_e8),
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

print("Training batches:", len(train_loader_e8))
print("Validation batches:", len(val_loader_e8))

Device: cpu
Training batches: 225
Validation batches: 29


In [8]:
import torch
import torch.nn as nn

class SignalCNN(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(12, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((2, 8))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 2 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [9]:
import copy
import random
import torch.nn as nn
from sklearn.metrics import f1_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

num_classes = len(label_encoder.classes_)

model_e8 = SignalCNN(num_classes=num_classes).to(device)

# Match the class weighting used in E5.
class_counts = np.bincount(
    y_encoded[train_idx],
    minlength=num_classes
)
class_weights_e8 = 1.0 / np.sqrt(class_counts)
class_weights_e8 /= class_weights_e8.mean()

criterion_e8 = nn.CrossEntropyLoss(
    weight=torch.tensor(
        class_weights_e8,
        dtype=torch.float32,
        device=device
    )
)

optimizer_e8 = torch.optim.AdamW(
    model_e8.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("Model parameters:", sum(
    p.numel() for p in model_e8.parameters()
))
print("Input channels:", model_e8.features[0].in_channels)

Model parameters: 359721
Input channels: 12


In [10]:
best_f1_e8 = -1.0
best_state_e8 = None
patience_e8 = 4
no_improve_e8 = 0
max_epochs_e8 = 20

for epoch in range(max_epochs_e8):
    model_e8.train()
    total_loss = 0.0
    total_samples = 0

    for xb, yb in train_loader_e8:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer_e8.zero_grad(set_to_none=True)

        logits = model_e8(xb)
        loss = criterion_e8(logits, yb)

        loss.backward()
        optimizer_e8.step()

        total_loss += loss.item() * len(yb)
        total_samples += len(yb)

    model_e8.eval()
    true_labels, predicted_labels = [], []

    with torch.no_grad():
        for xb, yb in val_loader_e8:
            xb = xb.to(device, non_blocking=True)
            logits = model_e8(xb)

            predicted_labels.extend(
                logits.argmax(dim=1).cpu().numpy()
            )
            true_labels.extend(yb.numpy())

    val_f1_e8 = f1_score(
        true_labels,
        predicted_labels,
        average="macro",
        zero_division=0
    )

    avg_loss = total_loss / total_samples

    print(
        f"Epoch {epoch + 1:02d}/{max_epochs_e8} | "
        f"Train loss: {avg_loss:.4f} | "
        f"Val macro-F1: {val_f1_e8:.4f}"
    )

    if val_f1_e8 > best_f1_e8:
        best_f1_e8 = val_f1_e8
        best_state_e8 = copy.deepcopy(model_e8.state_dict())
        no_improve_e8 = 0
        torch.save(best_state_e8, "e8_best.pt")
        print("  Best E8 checkpoint saved.")
    else:
        no_improve_e8 += 1

    if no_improve_e8 >= patience_e8:
        print("Early stopping.")
        break

model_e8.load_state_dict(best_state_e8)

print(f"\nBest E8 validation macro-F1: {best_f1_e8:.4f}")

Epoch 01/20 | Train loss: 1.4809 | Val macro-F1: 0.0977
  Best E8 checkpoint saved.
Epoch 02/20 | Train loss: 1.0813 | Val macro-F1: 0.4550
  Best E8 checkpoint saved.
Epoch 03/20 | Train loss: 0.9433 | Val macro-F1: 0.5208
  Best E8 checkpoint saved.
Epoch 04/20 | Train loss: 0.8684 | Val macro-F1: 0.2072
Epoch 05/20 | Train loss: 0.7822 | Val macro-F1: 0.1685
Epoch 06/20 | Train loss: 0.7040 | Val macro-F1: 0.1780
Epoch 07/20 | Train loss: 0.6505 | Val macro-F1: 0.1896
Early stopping.

Best E8 validation macro-F1: 0.5208
